<a href="https://colab.research.google.com/github/l-yunxi/optimisation-multiclass-regression/blob/main/homework_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Homework Project

In [ ]:
import numpy as np
from scipy.special import logsumexp
import matplotlib.pyplot as plt

### Question 1

In [ ]:
np.random.seed(42)  # the answer to everything obviously

d, m, k = 1000, 1000, 50

# A is our data matrix
A = np.random.randn(m, d)

#print(A.shape)
print(A[:3, :3])

[[ 0.49671415 -0.1382643   0.64768854]
 [ 1.39935544  0.92463368  0.05963037]
 [-0.67517827 -0.14451867 -0.79241992]]


### Question 2

In [ ]:
# generate matrix X and E
# E is noise
# X is our parameter/weights matrix
X = np.random.randn(d, k)
E = np.random.randn(m, k)

print(X.shape)

Y = A @ X + E

#index of the maximum value in each row - this is the class label
b = np.argmax(Y, axis = 1)


(1000, 50)


### Loss Function

In [ ]:
# loss function

def compute_loss(x, A, b, m):
  scores = A @ x
  c_score = scores[np.arange(m), b] # take the correct score for the class
  log_a = logsumexp(scores, axis = 1) # compute log... for every row/object
  per_ob = -c_score + log_a   # cross-entropy loss per object
  return np.sum(per_ob)

### Partial Deriv Function

In [ ]:
# partial derivative of loss fn

def partial_deriv(x, A, m, b, k):
  scores = A @ x
  scores -= np.max(scores, axis = 1, keepdims = True) # I did it to avoid overflow
  exp_s = np.exp(scores)
  sumexp_s = np.sum(exp_s, axis = 1, keepdims=True)
  soft_max = exp_s / sumexp_s # probability the given example is in the right class
  I = np.zeros((m, k))
  I[np.arange(m), b] = 1.0 # indicator function as a matrix
  err_matrix = I - soft_max
  samples_sum = A.T @ err_matrix # summing all samples i while weighting by their feature
  return -1 * samples_sum

#print(partial_deriv(X)) - synthetic data
#print(partial_deriv(X_init, A_train, m1, b_train, k1))


### Gradient Descent Code

In [ ]:
# implementing gradient descent

#X^(t+1) = X^(t) - stepsize*grad(f(X^(t))), where t is each step
start_point = np.zeros((d, k))
learn_rate = 0.01    # I changed learning rate to the optimal for now - from 0.1 to 0.01
iterations = 100
tolerance = 1e-2 # 0.01

# lipshitz for gradient descent, but doesn't work properly yet, need to be adjusted
#L_gd = np.linalg.norm(A, ord = 2)**2
#learn_rate = 1 / L_gd

def grad_descent(start_point, learn_rate, iterations, tol):
    x = start_point
    prev_loss = float('inf')

    for i in range(iterations):
        x = x - learn_rate * partial_deriv(x, A, m, b, k)  # update step
        cur_loss = float(compute_loss(x, A, b, m))
        print(f"Iteration {i+1}: compute_loss(x) = {float(cur_loss):.4f}")

    # stopping condition
        #print(f"{abs(cur_loss - prev_loss): .6f}")
        if abs(cur_loss - prev_loss) < tol:
            print(f"Converged at: {i + 1}")
            break

        prev_loss = cur_loss

    return x

print(grad_descent(start_point, learn_rate, iterations, tolerance).shape)

#minimum = grad_descent(start_point, learn_rate, iterations)
#print(minimum)
#print(f"\nLocal minimum occurs at x = {float(minimum):.4f}, f(x) = {float(partial_deriv(minimum)):.4f}")

Iteration 1: compute_loss(x) = 7.6851
Iteration 2: compute_loss(x) = 5.3133
Iteration 3: compute_loss(x) = 4.6644
Iteration 4: compute_loss(x) = 4.2325
Iteration 5: compute_loss(x) = 3.9104
Iteration 6: compute_loss(x) = 3.6550
Iteration 7: compute_loss(x) = 3.4446
Iteration 8: compute_loss(x) = 3.2665
Iteration 9: compute_loss(x) = 3.1126
Iteration 10: compute_loss(x) = 2.9777
Iteration 11: compute_loss(x) = 2.8579
Iteration 12: compute_loss(x) = 2.7504
Iteration 13: compute_loss(x) = 2.6532
Iteration 14: compute_loss(x) = 2.5647
Iteration 15: compute_loss(x) = 2.4835
Iteration 16: compute_loss(x) = 2.4087
Iteration 17: compute_loss(x) = 2.3395
Iteration 18: compute_loss(x) = 2.2752
Iteration 19: compute_loss(x) = 2.2152
Iteration 20: compute_loss(x) = 2.1591
Iteration 21: compute_loss(x) = 2.1064
Iteration 22: compute_loss(x) = 2.0567
Iteration 23: compute_loss(x) = 2.0099
Iteration 24: compute_loss(x) = 1.9656
Iteration 25: compute_loss(x) = 1.9237
Iteration 26: compute_loss(x) = 1.

### Partial Deriv with Caching

In [ ]:
# partial derivative with caching

def partial_deriv_bcgd(scores, A, m, k, b):
  scores -= np.max(scores, axis = 1, keepdims = True) # I did it to avoid overflow
  exp_s = np.exp(scores)
  sumexp_s = np.sum(exp_s, axis = 1, keepdims=True)
  soft_max = exp_s / sumexp_s
  I = np.zeros((m, k))
  I[np.arange(m), b] = 1.0
  err_matrix = I - soft_max
  samples_sum = A.T @ err_matrix
  return -1 * samples_sum

### BCGD Loss Function

In [ ]:
def compute_loss_bcgd(scores, b, m):
  c_score = scores[np.arange(m), b] # take the correct score for the class
  log_a = logsumexp(scores, axis = 1) # compute log... for every row/object
  per_ob = -c_score + log_a   # cross-entropy loss per object
  return np.sum(per_ob) # loss for all

### BCGD

In [ ]:
 # implementing block coordinate descent (BCD) with Gauss–Southwell rule



# DONE TODO: add lipshitz constant - to customize stepsize
# DONE TODO 2: cut the data to sets
# Not necessary?? TODO 3: create heat map of weights
# ToDo 4: create loss curve (train & val)
# DONE TODO 5: what about stopping condition?

#learn_rate2 = 0.05
it2 = 20000
tol2 = 1e-4
start_point2 = np.zeros((d, k))


#print(start_point2.sum())

def bcgd(st_point, it, tol):
    x = st_point.copy()
    scores = A @ x
    prev_loss = float('inf')
    eta = 1.7

    for i in range(it):
      #if (it % 1500 == 0):
      #  eta *= 0.8
      grad = partial_deriv_bcgd(scores)
      j = np.argmax(np.linalg.norm(grad, axis=1))
      # print(f"Upd row:{j}, block_size {grad[j].shape}")

      L_j = np.linalg.norm(A[:,j]) **2 # L_i

      step = eta / L_j * grad[j]  # α_i * f = (1/L_i) * grad
      #step = lr * grad[j]
      x[j] -= step  # y_i = y_{i-1} - α_i * grad
      scores -= np.outer(A[:,j], step)

      cur_loss = float(compute_loss_bcgd(scores))
      print(f"Iteration {i+1}: compute_loss(x) = {float(cur_loss):.4f}")

      # stopping condition
      #print(f"{abs(cur_loss - prev_loss): .6f}")
      if abs(cur_loss - prev_loss) < tol:
        print(f"Converged at: {i + 1}")
        break

      prev_loss = cur_loss
    return x

print(bcgd(start_point2, it2, tol2).shape)


TypeError: partial_deriv_bcgd() missing 4 required positional arguments: 'A', 'm', 'k', and 'b'

### Importing Data

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/forest-cover-type-dataset")

print("Path to dataset files:", path)

100%|██████████| 11.2M/11.2M [00:00<00:00, 85.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/uciml/forest-cover-type-dataset/versions/1


In [ ]:
!pip install scikit-learn

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [ ]:
import gdown

id = '1vRgxjLpGQlmX1S8z8LeB8HnMWkC5KwIl'
gdown.download(f'https://drive.google.com/uc?id={id}', 'covtype.csv')


Downloading...
From: https://drive.google.com/uc?id=1vRgxjLpGQlmX1S8z8LeB8HnMWkC5KwIl
To: /content/covtype.csv
100%|██████████| 75.2M/75.2M [00:00<00:00, 119MB/s]


'covtype.csv'

In [ ]:
   # importing the data

df = pd.read_csv('covtype.csv')
df = df.fillna(0)
df = df[df['Cover_Type'] != 0]
#print(df['Cover_Type'].unique()) #ensuring
#df.isnull().sum()

### Cleaning Data

In [ ]:
   # shuffle rows and convert into matrix

res = df.sample(frac=1).reset_index(drop=True)
A1 = res.drop(columns=['Cover_Type']).values
b1 = res['Cover_Type'].values # our classes


   # split data - train, val and test sets (80 - 10 - 10)

#for A1
split_1 = int(0.8 * len(A1))
A_train = A1[:split_1]                 # training - A
A_val_test = A1[split_1:]

split_2 = int(0.5 * len(A_val_test))
A_val = A_val_test[:split_2]           # validation - A
A_test = A_val_test[split_2:]          # test - A

#for b1
b_train = b1[:split_1]                 # training - B
b_val_test = b1[split_1:]

split_2b = int(0.5 * len(b_val_test))
b_val = b_val_test[:split_2b]          # validation - B
b_test = b_val_test[split_2b:]         # test - B

   # scale the training set and transform val and test set

scaler = StandardScaler()

#for A1
scal_A_train = scaler.fit_transform(A_train)  # scales and transforms
trans_A_val   = scaler.transform(A_val)       # only transforms
trans_A_test  = scaler.transform(A_test)      # only transforms



### check

In [ ]:
print(b_train.shape)
print(b_val.shape)
print(b_test.shape)
print(b1.shape)


In [ ]:
print(A_train.shape)
print(A_val.shape)
print(A_test.shape)
print(A1.shape)

### Test on a real data

In [ ]:
# to change indexes of the labels from 1..7 to 0..6
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
b_train = le.fit_transform(b_train)
b_test = le.fit_transform(b_test)

#print(b_train.dtype)
#np.unique(b_train)

In [ ]:
# making new X matrix (d1, k1)
np.random.seed(13)
d1 = scal_A_train.shape[1]
k1 = len(np.unique(b_train))
m1 = len(scal_A_train)
X_init = np.zeros((d1, k1))

# check if all objects have labels
print(np.sum(b_train == -1))
len(b_train) == len(A_train)

0


True

In [ ]:
print(compute_loss(X_init, A_train, b_train, m1))

904476.5504722501


In [ ]:
print((partial_deriv(X_init, A_train, m1, b_train, k1))[:3,:3])
print((partial_deriv(X_init, scal_A_train, m1, b_train, k1))[:3,:3])

[[-3.33826398e+08 -4.64960781e+08  1.27887301e+08]
 [-1.61040893e+07 -2.40947263e+07  5.28622671e+06]
 [-1.28869571e+06 -2.13335371e+06  3.42851286e+05]]
[[-102423.19352778   30998.44926139   57736.53625441]
 [   -546.84661443    7223.60769079   -5249.28325495]
 [  22156.32606718   16649.01062365  -25320.99196774]]


### Real Data Gradient Descent

In [ ]:
# implementing gradient descent - non scaled

start_point = X_init
learn_rate = 1e-13
iterations = 1000
tol = 5          # 1e-6


def grad_descent(start_point, learn_rate, iterations, tol):
    x = start_point
    prev_loss = float('inf')

    for i in range(iterations):
        x = x - learn_rate * partial_deriv(x, A_train, m1, b_train, k1)  # update step
        cur_loss = float(compute_loss(x, A_train, b_train, m1))
        print(f"Iteration {i+1}: compute_loss(x) = {float(cur_loss):.4f}")

    # stopping condition
        if abs(cur_loss - prev_loss) < tol:
            print(f"Converged at: {i + 1}")
            break

        prev_loss = cur_loss

    return x

print(grad_descent(start_point, learn_rate, iterations, tol).shape)

Iteration 1: compute_loss(x) = 807962.6824
Iteration 2: compute_loss(x) = 738189.8829
Iteration 3: compute_loss(x) = 689258.3318
Iteration 4: compute_loss(x) = 654969.9059
Iteration 5: compute_loss(x) = 630497.0139
Iteration 6: compute_loss(x) = 612576.0435


KeyboardInterrupt: 

In [ ]:
# implementing gradient descent - scaled

start_point = X_init
learn_rate = 1e-6
iterations = 1000
tol = 23  # 1e-6


def grad_descent_scaled(st_point, lr, it, tol):
  global X_checkpoint_gd
  ################# TIME ###############
  start_gd = time.time()
  acc_gd = []
  times_gd = []
  iteration_gd = []
  loss_gd = []
  ############## END TIME ##############

  x = st_point.copy()
  scores = A @ x

  prev_loss = float('inf')

  for i in range(it):
      x = x - lr * partial_deriv(x, scal_A_train, m1, b_train, k1)  # update step
      cur_loss = float(compute_loss(x, scal_A_train, b_train, m1))
      print(f"Iteration {i+1}: compute_loss(x) = {float(cur_loss):.4f}")

       # check accuracy + time + checkpoint
      if (i + 1) % 100 == 0:
        X_checkpoint_gd = x.copy()  # in the case I want to stop the function earlier
        pred = np.argmax(scores, axis = 1)
        accuracy = np.mean(pred == b_train)
        #print(f"loss = {cur_loss}, acc = {accuracy}")
        acc_gd.append(accuracy)  # approximate
        time_gd = time.time() - start_gd
        times_gd.append(time_gd)
        iteration_gd.append(i+1)
        loss.append(cur_loss)

    # stopping condition
      if abs(cur_loss - prev_loss) < tol:
          print(f"Converged at: {i + 1}")
          break

      prev_loss = cur_loss

  return x, acc_gd, times_gd, iteration_gd, loss_gd

NameError: name 'X_init' is not defined

### Accuracy Computing - GD

In [ ]:
 X_gd, acc_gd, times_gd, iteration_gd, loss_gd  = grad_descent_scaled(start_point, learn_rate, 50, tol)

In [ ]:

def predict(x, A):
  scores = A @ x
  return np.argmax(scores, axis=1)

predictions = predict(X_gd, trans_A_test)
accuracy = np.mean(predictions == b_test)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7042


In [ ]:
predictions = predict(X_gd, trans_A_test)
accuracy = np.mean(predictions == b_test)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7042


### Real Data BCGD

In [ ]:
it2 = 3000
# 23 between 1e-5 and 1e-4
tol2 = 23  # 5 that corresponds to 1e-5, because we have sum
start_point2 = np.zeros((d1, k1))

# 2 / 464809 ≈ 4.3e-6 on iteration ≈ 10000
#print(start_point2.sum())

def bcgd(st_point, it, tol, A, b):
    global X_checkpoint
    ################# TIME ###############
    start = time.time()
    acc_bcgd = []
    times = []
    iteration = []
    loss = []
    ############## END TIME ##############

    x = st_point.copy()
    scores = A @ x

    prev_loss = float('inf')
    eta = 1

    for i in range(it):
     # if (it % 1500 == 0):
      #  eta *= 0.8

      grad = partial_deriv_bcgd(scores, scal_A_train, m1, k1, b_train)
      j = np.argmax(np.linalg.norm(grad, axis=1)) # take the highest gradient
      # print(f"Upd row:{j}, block_size {grad[j].shape}")

      # if you don`t see the square here it means that google collab deleted it again **2
      L_j = np.linalg.norm(A[:,j])**2 # L_j - Lipschitz constant for block j

      step = eta / L_j * grad[j]  # α_j * f = (1/L_j) * grad
      #step = lr * grad[j]  # for without L
      x[j] -= step  # x_j = x[j-1] - α_j * grad
      scores -= np.outer(A[:,j], step) # A[:,j] * step

      cur_loss = float(compute_loss_bcgd(scores,b_train, m1))
      print(f"Iteration {i+1}: compute_loss(x) = {float(cur_loss):.4f}")

      # check accuracy + time + checkpoint
      if (i + 1) % 100 == 0:
        X_checkpoint = x.copy()  # in the case I want to stop the function earlier
        pred = np.argmax(scores, axis = 1)
        accuracy = np.mean(pred == b_train)
        #print(f"loss = {cur_loss}, acc = {accuracy}")
        acc_bcgd.append(accuracy)  # approximate
        time_bcgd = time.time() - start
        times.append(time_bcgd)
        iteration.append(i+1)
        loss.append(cur_loss)

      # stopping condition
      #print(f"{abs(cur_loss - prev_loss): .6f}")
      if abs(cur_loss - prev_loss) < tol:
        print(f"Converged at: {i + 1}")
        break

      prev_loss = cur_loss

    return x, acc_bcgd, times, iteration, loss

#### Computing time + prediction


In [ ]:
import time

In [ ]:
X_BCGD, acc_bcgd, times, iteration_bcgd, loss = bcgd(start_point2, it2, tol2, scal_A_train, b_train)


Iteration 1: compute_loss(x) = 872224.0778
Iteration 2: compute_loss(x) = 848058.1113
Iteration 3: compute_loss(x) = 829592.4971
Iteration 4: compute_loss(x) = 815307.5818
Iteration 5: compute_loss(x) = 802586.5157
Iteration 6: compute_loss(x) = 791921.4030
Iteration 7: compute_loss(x) = 783117.4421
Iteration 8: compute_loss(x) = 774875.4471
Iteration 9: compute_loss(x) = 767985.8405
Iteration 10: compute_loss(x) = 761496.5249
Iteration 11: compute_loss(x) = 755653.8701
Iteration 12: compute_loss(x) = 750496.8815
Iteration 13: compute_loss(x) = 745310.2424
Iteration 14: compute_loss(x) = 740517.0594
Iteration 15: compute_loss(x) = 736001.1679
Iteration 16: compute_loss(x) = 732046.8717
Iteration 17: compute_loss(x) = 727764.4658
Iteration 18: compute_loss(x) = 723647.1780
Iteration 19: compute_loss(x) = 719951.0250
Iteration 20: compute_loss(x) = 716010.1988
Iteration 21: compute_loss(x) = 712184.8875
Iteration 22: compute_loss(x) = 708462.0612
Iteration 23: compute_loss(x) = 704832.44

In [ ]:
# Work of 2 hours
x_save = X_BCGD.copy()
acc_save = acc_bcgd.copy()
times_save = times.copy()

In [ ]:
def predict_BCGD(x, A):
  scores = A @ x
  return np.argmax(scores, axis=1)

In [ ]:
predictions_bcgd = predict_BCGD(X_checkpoint, trans_A_test)
accuracy_test = np.mean(predictions_bcgd == b_test)

# TODO: check on validation set

print(accuracy_test)

In [ ]:
print((partial_deriv(X_init, A_train, m1, b_train, k1))[:3,:3])
print((partial_deriv(X_init, scal_A_train, m1, b_train, k1))[:3,:3])

### TEST TIME PLOT


In [ ]:
# TODO:
# - plot accuracy vs iteration
# - loss vs iteration

In [ ]:
# plot on 20000 iterations
plt.plot(times, acc_bcgd, label = 'BCGD')
plt.xlabel('Time')
plt.ylabel('Accuracy')
plt.title('Accuracy vs CPU time')
plt.legend()

NameError: name 'times' is not defined

In [ ]:
# plot on 20000 iterations
plt.plot(times_gd, acc_gd, label = 'GD')
plt.xlabel('Time')
plt.ylabel('Accuracy')
plt.title('Accuracy vs CPU time')
plt.legend()

NameError: name 'times_gd' is not defined

#### Accuracy vs CPU time

In [ ]:
# add gradient descent line

plt.plot(times, acc_bcgd, label = 'BCGD')
plt.xlabel('Time')
plt.ylabel('Accuracy')
plt.title('Accuracy vs CPU time')
plt.legend()

#### Accuracy vs Iteration

In [ ]:


plt.plot(iteration_bcgd, acc_bcgd,  label = 'BCGD')
plt.xlabel('Iteration')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Iteration')
plt.legend()

#### Loss vs Iteration


In [ ]:
plt.plot(iteration_bcgd, loss, label = 'BCGD')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Loss vs Iteration')